# 同源开盘收益分解
固定1056条训练样本，将旧日线→执行收益的对数差拆为同源09:30→09:47时间差与同源09:30→旧日线残差。不改变任何标签，不选择参数。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.data.open_clock_marks import build_open_clock_marks
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/open_clock_decomposition_20261015';parent=root/'research/execution_return_training_20261015'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
plan=json.loads((root/'research/full_path_tiered_allocation_20261013/plan.json').read_text())
quotes,manifest=build_open_clock_marks(source_manifest=root/plan['source_manifest'],source_sha256=plan['source_id'],provider_uri=root/'data/qlib/open_clock_decomposition_20261015',as_of=plan['as_of'],workspace_root=root)
qlib.init(provider_uri=manifest['provider_uri'],region='us')
s=json.loads((parent/'summary.json').read_text());inventory=json.loads((parent/'verification_manifest.json').read_text())
for e in inventory['entries']:
    if e['path'].endswith(('inputs/original_labels','inputs/execution_labels')):assert hashlib.sha256((parent/e['path']).read_bytes()).hexdigest()==e['sha256']
with R.uri_context(s['recorder_uri']):
    rec=R.get_recorder(recorder_id=s['recorder_id'],experiment_id=s['experiment_id']);old=rec.load_object('inputs/original_labels');new=rec.load_object('inputs/execution_labels')
rows=[];details={}
for symbol in ['SOXL','SOXX']:
    asset=symbol.lower();k=asset+'_forward_open_return';idx=old.loc['2018-10-12':'2022-12-21'].dropna(subset=[k]).index
    q=quotes.xs(symbol,level='instrument');q=q.rename(columns=lambda x:x.removeprefix('$'))
    entry=q.reindex(pd.DatetimeIndex(old.loc[idx,'trade_date'])).set_axis(idx);exit=q.reindex(pd.DatetimeIndex(old.loc[idx,'exit_date'])).set_axis(idx)
    assert entry.open947.notna().all() and exit.open947.notna().all()
    z=pd.DataFrame({'old_log':np.log1p(old.loc[idx,k]),'native_log':np.log1p(new.loc[idx,k]),'same_source_open_log':np.log(exit.open930/entry.open930),'same_source_execution_log':np.log(exit.open947/entry.open947)},index=idx)
    assert np.max(np.abs(z.native_log-z.same_source_execution_log))<2e-7
    z['clock']=z.same_source_execution_log-z.same_source_open_log
    z['source_basis_residual']=z.same_source_open_log-z.old_log
    z['total']=z.native_log-z.old_log
    z['adjustment_event']=entry.affine_a.ne(exit.affine_a)|entry.affine_b.ne(exit.affine_b)|entry.split_factor.ne(exit.split_factor)
    z['decomposition_known']=z[['clock','source_basis_residual']].notna().all(axis=1)
    assert np.max(np.abs((z.total-z.clock-z.source_basis_residual).dropna()))<2e-7
    details[symbol]=z
    print(symbol,'retained',len(z),'unknown',int((~z.decomposition_known).sum()),'unknown_dates',z.index[~z.decomposition_known].astype(str).tolist())
    for scope,g in [('all_known',z[z.decomposition_known]),('event_known',z[z.adjustment_event & z.decomposition_known]),('non_event_known',z[~z.adjustment_event & z.decomposition_known])]:
        total=g.total.to_numpy(float);clock=g.clock.to_numpy(float);res=g.source_basis_residual.to_numpy(float)
        rows.append(dict(symbol=symbol,scope=scope,n=len(g),total_rms_bps=1e4*np.sqrt(np.mean(total**2)),clock_rms_bps=1e4*np.sqrt(np.mean(clock**2)),source_basis_residual_rms_bps=1e4*np.sqrt(np.mean(res**2)),clock_projection_share=np.dot(clock,total)/np.dot(total,total),residual_projection_share=np.dot(res,total)/np.dot(total,total),residual_max_abs_bps=1e4*np.max(np.abs(res))))
table=pd.DataFrame(rows)
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
    rec=R.get_recorder();rec.save_objects(artifact_path='inputs',quotes=quotes,manifest=manifest,source_inventory=inventory,builder=(root/'src/data/open_clock_marks.py').read_text());rec.save_objects(artifact_path='analysis',decomposition=details,metrics=table);rec.log_metrics(training_rows=len(idx),registered_artifact_count=6)
summary={'experiment':card.name,'decision':'diagnostic_only_no_promotion','new_fits':0,'new_backtests':0,'source_recorder':s['recorder_id'],'provider_uri':manifest['provider_uri'],'region':'us','as_of':plan['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'artifact_policy':policy,'scope':'Exact log-return decomposition on original mature training dates; projection shares are algebraic covariance attribution, not causal alpha or independent source certification.'}
table.to_csv(card/'decomposition_metrics.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':manifest['provider_uri'],'artifact_policy':policy}))
binding=json.loads((parent/'nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2))
a=card/'qlib_recorder'/str(rec.experiment_id)/rec.id/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(a.rglob('*')) if p.is_file()]},indent=2))
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(table.to_string(index=False))


# 同源分解：主要是开盘区间差异，未证明新特征有效

本轮0拟合、0新增回测。使用同一冻结NONE分钟来源及相同公开仿射调整系数，将09:30/09:47开盘标记写入Qlib日频诊断store，经D.features读回，再与前次1056条成熟训练标签逐日核验。日频仅为诊断存储粒度，字段不是当日开盘即可使用的预测特征。未修改原始价格、标签或账户。

对数收益严格分解为：执行标签减旧日线标签 = 同源09:47收益减同源09:30收益 + 同源09:30收益减旧日线收益。重构误差小于2e-7，09:47收益与现有执行标签也在该精度内一致。

| 标的 | 总训练行 / 可分解行 | 总差值RMS | 同源时点差RMS | 来源/调整残差RMS |
|---|---:|---:|---:|---:|
| SOXL | 1056 / 1056 | 313.168 bp | 313.169 bp | 0.042 bp |
| SOXX | 1056 / 1054 | 101.448 bp | 101.448 bp | 0.009 bp |

最大来源/调整残差分别0.499 bp和0.089 bp。有效样本中，差异几乎全部来自两个交易日开盘至09:47区间的变化，而非日线与分钟来源/调整口径。分解是代数恒等及同期一致性，不是预测IC或因果alpha，不证明任一来源具有独立价格真值。公司行动调整也未认证真实派息现金流或历史版本PIT；事件分组仅按相邻日期调整算子变化识别。

门禁发现已有记录中的SOXX 2020-03-16首分钟OHLC矛盾：open/high/low=188、close=209.42。原始记录仍未修复。初次执行在0拟合时终止；随后明确改为诊断unknown合同，保留该原始记录与全部1056训练日期，无效0930标记在Qlib中为NaN并附valid930=0，两条受影响标签仅无法分解，没有从模型训练或回测中删除。09:47标记本身有效。该2条不支持任何关于开盘时点归因的结论。

下一项有界训练假设：原输入以过去交易日信息为主，预测时段改为09:47以后，却未加入当日已经观察到的开盘状态；可测试同源隔夜跳空及09:30至09:45开盘收益是否有条件增量。须固定字段组、相同缺失标记对照、原执行收益标签、相同seed/训练轮数与0/2/5bp完整账户，并保留长期强参考；缺失源值不可填造，09:47成交价及其后数据不能进入当次输入。只有通过配对训练和完整账户风险门才可判断有效。本轮不批准或宣称新候选有效，也不改生产；稳定超额目标未完成。

Qlib Recorder及NAS摘要见summary.json，逐对象冷恢复验证见readback_verification.json。方法依据官方Qlib Data文档：https://qlib.readthedocs.io/en/latest/component/data.html 。
